In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

In [2]:
import jax
# Enable float64 support in JAX before importing jax.numpy
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import time

In [3]:


# ==========================================
# 1. The Analytical Solver (Cardano / Trigonometric)
# ==========================================
def analytical_solve(K0, K1, K2, K3):
    """
    Solves K3*d^3 + K2*d^2 + K1*d - K0 = 0 using the robust Cardano method.
    """
    d_term = -K0
    c_term = K1
    b_term = K2
    a_term = K3

    # Quadratic Fallback
    discriminant_q = c_term**2 - 4 * b_term * d_term
    sqrt_disc_q = jnp.sqrt(jnp.maximum(0.0, discriminant_q))
    q_q = -0.5 * (c_term + jnp.sign(c_term) * sqrt_disc_q)
    
    safe_b = jnp.where(jnp.abs(b_term) > 1e-12, b_term, 1.0)
    safe_q_q = jnp.where(jnp.abs(q_q) > 1e-12, q_q, 1.0)
    
    root0_q = jnp.where(jnp.abs(b_term) > 1e-12, q_q / safe_b, -d_term / (c_term + 1e-12))
    root1_q = jnp.where(jnp.abs(b_term) > 1e-12, d_term / safe_q_q, root0_q)
    
    r1_quad = jnp.where(discriminant_q >= 0, root0_q, jnp.inf)
    r2_quad = jnp.where(discriminant_q >= 0, root1_q, jnp.inf)
    r3_quad = jnp.inf

    # Cubic Case
    safe_a = a_term + jnp.sign(a_term)*1e-12
    B = b_term / safe_a
    C = c_term / safe_a
    D = d_term / safe_a

    Q = (B**2 - 3.0*C) / 9.0
    R = (2.0 * B**3 - 9.0 * B * C + 27.0 * D) / 54.0

    safe_Q = jnp.maximum(Q, 0.0)
    is_multi = (R**2 < Q**3)

    # Branch A: 3 real roots
    arccos_input = jnp.clip(R / jnp.sqrt(safe_Q**3 + 1e-12), -1.0, 1.0)
    theta = jnp.arccos(arccos_input)
    
    sqrt_Q = jnp.sqrt(safe_Q)
    t1_multi = -2.0 * sqrt_Q * jnp.cos(theta / 3.0) - B / 3.0
    t2_multi = -2.0 * sqrt_Q * jnp.cos((theta + 2.0 * jnp.pi) / 3.0) - B / 3.0
    t3_multi = -2.0 * sqrt_Q * jnp.cos((theta - 2.0 * jnp.pi) / 3.0) - B / 3.0
    
    # Branch B: 1 real root
    diff = jnp.maximum(R**2 - Q**3, 0.0)
    sqrt_diff = jnp.sqrt(diff)
    abs_R_plus_sqrt = jnp.abs(R) + sqrt_diff
    E = -jnp.sign(R) * jnp.cbrt(abs_R_plus_sqrt)
    E_is_valid = jnp.abs(E) > 1e-12
    safe_E = jnp.where(E_is_valid, E, 1.0)
    F = (Q / safe_E) * E_is_valid

    t1_single = E + F - B / 3.0
    t2_single = jnp.inf
    t3_single = jnp.inf

    r1_cubic = jnp.where(is_multi, t1_multi, t1_single)
    r2_cubic = jnp.where(is_multi, t2_multi, t2_single)
    r3_cubic = jnp.where(is_multi, t3_multi, t3_single)

    # Merge
    is_cubic = jnp.abs(a_term) > 1e-12
    r1_merged = jnp.where(is_cubic, r1_cubic, r1_quad)
    r2_merged = jnp.where(is_cubic, r2_cubic, r2_quad)
    r3_merged = jnp.where(is_cubic, r3_cubic, r3_quad)

    r1_pos = jnp.where(r1_merged > 0, r1_merged, jnp.inf)
    r2_pos = jnp.where(r2_merged > 0, r2_merged, jnp.inf)
    r3_pos = jnp.where(r3_merged > 0, r3_merged, jnp.inf)

    return jnp.minimum(jnp.minimum(r1_pos, r2_pos), r3_pos)

# ==========================================
# 2. The Unrolled Newton-Raphson Solver
# ==========================================
def newton_solve(K0, K1, K2, K3, steps):
    b1 = K0 / (K1 + 1e-12)
    b2 = jnp.sqrt(K0 / (K2 + 1e-12))
    b3 = jnp.cbrt(K0 / (K3 + 1e-12))
    d = jnp.minimum(jnp.minimum(b1, b2), b3)
    
    for _ in range(steps):
        val = d * (K1 + d * (K2 + d * K3)) - K0
        deriv = K1 + d * (2.0 * K2 + d * 3.0 * K3)
        d = d - val / (deriv + 1e-12)
    return d



In [9]:
def run_benchmarks():
    N_RAYS = 10_000_000
    print(f"Initializing benchmark for {N_RAYS:,} bounds calculations...\n")
    
    key = jax.random.PRNGKey(42)
    
    for dtype, dtype_name in [(jnp.float32, "Float32"), (jnp.float64, "Float64")]:
        print(f"========================================")
        print(f"       Precision: {dtype_name}")
        print(f"========================================")
        
        key, subkey = jax.random.split(key)
        K = jax.random.uniform(subkey, (4, N_RAYS), dtype=dtype, minval=0.001, maxval=10.0)
        K0, K1, K2, K3 = K[0], K[1], K[2], K[3]

        # ----------------------------------------
        # Time Analytical Solver & Compute Residual
        # ----------------------------------------
        @jax.jit
        def run_analytical(k0, k1, k2, k3):
            return jax.vmap(analytical_solve)(k0, k1, k2, k3)
        
        print("JIT Compiling Analytical...")
        _ = run_analytical(K0[:10], K1[:10], K2[:10], K3[:10])
        
        start = time.perf_counter()
        d_analytical = run_analytical(K0, K1, K2, K3).block_until_ready()
        t_analytical = time.perf_counter() - start
        
        # Calculate Analytical Residual
        valid_mask_a = jnp.isfinite(d_analytical)
        res_a = jnp.abs(d_analytical * (K1 + d_analytical * (K2 + d_analytical * K3)) - K0)
        max_res_a = jnp.max(jnp.where(valid_mask_a, res_a, 0.0))
        mean_res_a = jnp.mean(jnp.where(valid_mask_a, res_a, 0.0))
        
        print(f"Analytical Runtime      : {t_analytical:.4f} seconds")
        print(f"Analytical Max Residual : {max_res_a:.2e}")
        print(f"Analytical Mean Residual: {mean_res_a:.2e}\n")

        # ----------------------------------------
        # Time Newton Solver & Compute Residuals
        # ----------------------------------------
        print(f"{'Steps':<5} | {'Runtime':<8} | {'Speedup':<8} | {'Max Diff':<9} | {'Newton Max Res':<15} | {'Newton Mean Res':<15}")
        print("-" * 75)
        
        for steps in [1, 2, 3, 4, 5]:
            @jax.jit
            def run_newton(k0, k1, k2, k3):
                return jax.vmap(lambda a,b,c,d: newton_solve(a,b,c,d, steps))(k0, k1, k2, k3)
            
            _ = run_newton(K0[:10], K1[:10], K2[:10], K3[:10])
            
            start = time.perf_counter()
            d_newton = run_newton(K0, K1, K2, K3).block_until_ready()
            t_newton = time.perf_counter() - start
            
            # Calculate Differences
            diff = jnp.abs(d_analytical - d_newton)
            valid_mask_n = jnp.isfinite(d_newton) & valid_mask_a
            max_diff = jnp.max(jnp.where(valid_mask_n, diff, 0.0))
            
            # Calculate Newton Residual
            res_n = jnp.abs(d_newton * (K1 + d_newton * (K2 + d_newton * K3)) - K0)
            max_res_n = jnp.max(jnp.where(valid_mask_n, res_n, 0.0))
            mean_res_n = jnp.mean(jnp.where(valid_mask_n, res_n, 0.0))
            
            speedup = t_analytical / t_newton
            
            print(f"{steps:<5} | {t_newton:<8.4f} | {speedup:<6.2f}x | {max_diff:<9.2e} | {max_res_n:<15.2e} | {mean_res_n:<15.2e}")
        print("\n")

In [10]:

run_benchmarks()

Initializing benchmark for 10,000,000 bounds calculations...

       Precision: Float32
JIT Compiling Analytical...
Analytical Runtime      : 0.3493 seconds
Analytical Max Residual : 8.62e+00
Analytical Mean Residual: 1.19e-04

Steps | Runtime  | Speedup  | Max Diff  | Newton Max Res  | Newton Mean Res
---------------------------------------------------------------------------
1     | 0.0826   | 4.23  x | 8.02e-01  | 4.05e+00        | 6.87e-01       
2     | 0.0965   | 3.62  x | 4.90e-01  | 3.53e-01        | 3.45e-02       
3     | 0.1089   | 3.21  x | 4.91e-01  | 3.60e-03        | 1.56e-04       
4     | 0.1275   | 2.74  x | 4.91e-01  | 3.81e-06        | 2.50e-07       
5     | 0.1438   | 2.43  x | 4.91e-01  | 3.81e-06        | 2.06e-07       


       Precision: Float64
JIT Compiling Analytical...
Analytical Runtime      : 0.4096 seconds
Analytical Max Residual : 2.75e-08
Analytical Mean Residual: 1.16e-12

Steps | Runtime  | Speedup  | Max Diff  | Newton Max Res  | Newton Mean Res
-